# Mult-VAE: исследование на marketplace-домене

Аналог `ials_research.ipynb` для **Multinomial Variational AutoEncoder**
(Liang et al. 2018). Структура повторяет IALS-ноутбук, чтобы сравнение с
SVD/iALS было воспроизводимым:

1. Загрузка событий, inverse-frequency target.
2. Temporal split 80/20.
3. Базовый прогон с типовыми гиперпараметрами.
4. Optuna на 40%-подмножестве пользователей.
5. Финальная оценка лучшей конфигурации на полных данных.
6. Сравнение Mult-VAE / iALS / SVD.

In [1]:
import os
import pyarrow as pa     # импорт необходим, чтобы не крашился ноутбук на Windows
import numpy as np
import polars as pl
import torch

import src.dataset as dts
from src.modeling.vae import run_vae_experiment
from src.modeling.train_vae import build_user_history_dataset, train_vae
from src.config import VAE_DIR

print("torch:", torch.__version__)

2026-05-10 13:40:33.870 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: D:\HSE_repos\dreamteam-recsys


torch: 2.11.0+cu126


In [2]:
EVENTS_DIR = "../data/raw/dataset/small/marketplace/events"

events_df = pl.scan_parquet(f"{EVENTS_DIR}/*.pq", include_file_paths="path").sort("day")

In [3]:
# Inverse-frequency target — тот же, что в SVD/iALS-ноутбуках для конформности.
actions_count = (
    events_df.group_by("action_type")
    .agg(pl.len())
    .collect(engine="streaming")
    .with_columns(
        (pl.col("len").sum() / pl.col("len")).alias("target"),
        (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
        (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    )
    .drop("len")
)
events_df = events_df.join(actions_count.lazy(), on="action_type")
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""click""",34.582149,3.571844,5.880659
"""like""",3121.341812,8.046339,55.86897
"""view""",1.034082,0.710044,1.016898
"""clickout""",268.714913,5.597366,16.392526


In [11]:
train, test = dts.global_temporal_split(events_df, 0.2)
print(f"Train schema: {train.collect_schema()}")
print(f"Test schema:  {test.collect_schema()}")

Train schema: Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String, 'day': Int32, 'path': String, 'target': Float64, 'log_target': Float64, 'sqrt_target': Float64})
Test schema:  Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String, 'day': Int32, 'path': String, 'target': Float64, 'log_target': Float64, 'sqrt_target': Float64})


## Базовый прогон (OOM, GPU)

Архитектура `[600, 200]` — конфиг из оригинальной статьи Mult-VAE.
Бинаризованная история (`binary=True`) — стандартный режим для CF-VAE.

**Почему не `run_vae_experiment`?** На полном marketplace-сплите
(1.2M users × 498k items, 65M пар) `run_vae_experiment` строит
in-memory CSR + decoder ~498k → KILL'ом по VRAM на любой потребительской
карте. Поэтому baseline учим тем же OOM-стеком, что и CLI
(`python -m src.modeling.train_vae train`):

* feature-снимок `data/processed/datamart/features/events/user_id-item_id/{day}.pq`
  уже агрегирует 30-дневное окно → **20 000 selected items** и
  **~184 000 активных пользователей** (по `selected_items.pq` /
  `selected_users.pq`);
* sparse user-history стримится через `polars.sink_parquet`, дальше
  материализуется лениво HuggingFace `datasets` (Apache Arrow,
  memory-mapped);
* dense батч `(B, num_items)` собирается в коллатере **только** на текущий
  step и сразу освобождается.

**Параметры зеркалят успешный CLI-прогон**: `day=1305, epochs=30,
batch_size=256` (этот же набор стабильно работал на RTX 2080 SUPER 8GB).

> Target-leak: feature-снимок дня `D` агрегирует окно `[D-29, D]`,
> поэтому если соберётесь добавлять метрики поверх `test`, оценивать
> можно только события за дни `> D`.

In [5]:
import shutil
import tempfile
from pathlib import Path

DAY                = 1305          # снимок 30-дневного окна [1276, 1305]
EPOCHS             = 30
BATCH_SIZE         = 256
ENCODER_DIMS       = [600, 200]
DROPOUT            = 0.5
BETA               = 0.2
TOTAL_ANNEAL_STEPS = 200_000
KL_SCHEDULE        = "linear"
LR                 = 1e-3
WEIGHT_DECAY       = 0.0
BINARY             = True
MIN_USER_ITEMS     = 5
SEED               = 42

# Создаем временную папку для хранения истории
tmp_dir = Path(tempfile.mkdtemp(prefix="vae_history_"))
history_path = tmp_dir / "history.pq"
try:
    history_path, item_to_idx, user_to_idx = build_user_history_dataset(
        day=DAY,
        output_path=history_path,
        binary=BINARY,
        min_user_items=MIN_USER_ITEMS,
    )
    print(
        f"Снимок дня {DAY}: "
        f"{len(user_to_idx):,} активных users x {len(item_to_idx):,} items"
    )

    checkpoint_path = train_vae(
        history_path=history_path,
        item_to_idx=item_to_idx,
        user_to_idx=user_to_idx,
        artifacts_dir=VAE_DIR,
        encoder_dims=ENCODER_DIMS,
        dropout=DROPOUT,
        beta=BETA,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        total_anneal_steps=TOTAL_ANNEAL_STEPS,
        kl_schedule=KL_SCHEDULE,
        use_lr_scheduler=True,
        num_workers=0,            # Windows: иначе DataLoader крашится
        seed=SEED,
    )
    print(f"Чекпоинт сохранён: {checkpoint_path}")
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

2026-05-09 23:39:17.841 | INFO     | src.modeling.train_vae:build_user_history_dataset:142 - Сбор маппингов item/user...
2026-05-09 23:39:17.853 | INFO     | src.modeling.train_vae:build_user_history_dataset:154 - Каталог айтемов: 20000 штук
2026-05-09 23:39:17.856 | INFO     | src.modeling.train_vae:build_user_history_dataset:214 - Сохранение sparse user-history в C:\Users\Sasha\AppData\Local\Temp\vae_history_gy5h91dg\history.pq (streaming sink)...
2026-05-09 23:39:18.389 | INFO     | src.modeling.train_vae:build_user_history_dataset:223 - Активных пользователей: 184239
Снимок дня 1305: 184,239 активных users x 20,000 items
2026-05-09 23:39:18.693 | INFO     | src.modeling.vae:build_model:403 - MultiVAE собрана: num_items=20000, dims=[600, 200], device=cuda


Generating train split: 0 examples [00:00, ? examples/s]

2026-05-09 23:39:19.194 | INFO     | src.modeling.train_vae:train_vae:424 - Стартую обучение: 184239 пользователей, 20000 items, batches per epoch ≈ 720


Эпохи:   3%|▎         | 1/30 [00:17<08:34, 17.74s/epoch, beta=0.001, kld=295.267, loss=164.633, lr=9.97e-04, nll=164.499]

2026-05-09 23:39:49.342 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 01: loss=164.6326 nll=164.4987 kld=295.2670 beta=0.0007 lr=9.97e-04 time=17.7s


...

2026-05-09 23:50:05.097 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 28: loss=127.0821 nll=123.7139 kld=170.1155 beta=0.0202 lr=2.08e-05 time=28.2s


Эпохи:  97%|█████████▋| 29/30 [11:01<00:27, 27.72s/epoch, beta=0.021, kld=167.921, loss=127.134, lr=1.27e-05, nll=123.688]

2026-05-09 23:50:33.170 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 29: loss=127.1342 nll=123.6885 kld=167.9211 beta=0.0209 lr=1.27e-05 time=28.1s


Эпохи: 100%|██████████| 30/30 [11:29<00:00, 22.99s/epoch, beta=0.022, kld=166.140, loss=127.241, lr=1.00e-05, nll=123.712]


2026-05-09 23:51:01.359 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 30: loss=127.2409 nll=123.7121 kld=166.1397 beta=0.0216 lr=1.00e-05 time=28.2s
2026-05-09 23:51:01.599 | INFO     | src.modeling.vae:save:497 - MultiVAE сохранена в D:\HSE_repos\dreamteam-recsys\models\vae_artifacts\model.pt
2026-05-09 23:51:03.798 | INFO     | src.modeling.train_vae:train_vae:549 - user_items.npz сохранена в D:\HSE_repos\dreamteam-recsys\models\vae_artifacts
Чекпоинт сохранён: D:\HSE_repos\dreamteam-recsys\models\vae_artifacts\model.pt


## Гипероптимизация (Optuna)

Для сокращения затрат на ресурсы:
1. Берем **40% пользователей** случайной выборкой.
2. На подмножестве пользователей запускаем гипероптимизацию, целевая метрика — **NDCG@15**.
3. Лучшую конфигурацию переобучаем на полном датасете.

In [4]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SUBSET_FRACTION = 0.4
OPTUNA_TRIALS = 20
TOP_K = 15
RANDOM_SEED = 42

all_user_ids = (
    events_df.select("user_id").unique().collect(engine="streaming")["user_id"].to_list()
)
rng = np.random.default_rng(RANDOM_SEED)
subset_users = set(
    rng.choice(all_user_ids, size=int(len(all_user_ids) * SUBSET_FRACTION), replace=False).tolist()
)
print(f"Всего пользователей: {len(all_user_ids):,}")
print(f"Пользователей в подмножестве ({SUBSET_FRACTION*100:.0f}%): {len(subset_users):,}")

events_sub = events_df.filter(pl.col("user_id").is_in(subset_users))
train_sub, test_sub = dts.global_temporal_split(events_sub, 0.2)
print("Подмножество готово")

Всего пользователей: 1,616,763
Пользователей в подмножестве (40%): 646,705
Подмножество готово


In [6]:
events_df.head().collect()

timestamp,user_id,item_id,subdomain,action_type,os,day,path,target,log_target,sqrt_target
duration[μs],u64,str,str,str,str,i32,str,f64,f64,f64
1082d 89901µs,59328774,"""nfmcg_14777696""","""u2i""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898
1082d 163483µs,66295302,"""nfmcg_26098539""","""catalog""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898
1082d 864625µs,37542104,"""nfmcg_10840247""","""u2i""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898
1082d 889192µs,35193548,"""nfmcg_8040572""","""catalog""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898
1082d 936008µs,27256137,"""nfmcg_6770239""","""catalog""","""view""","""android""",1082,"""..\data\raw\dataset\small\mark…",1.034082,0.710044,1.016898


In [ ]:
ITEMS_TO_TAKE = 20_000

all_items_ids = (
    train_sub.group_by("item_id").agg(count=pl.len()).sort("count", descending=True).head(ITEMS_TO_TAKE).collect(engine="streaming")["item_id"].to_list()
)

In [6]:
train_sub = train_sub.filter(pl.col("item_id").is_in(all_items_ids))

Берем подмножество товаров для обучения:

In [7]:
import shutil
import tempfile
from pathlib import Path
from scipy.sparse import load_npz

from src.modeling.vae import MultiVAERecommender
from src.dataset import ndcg_at_k, precision_recall_at_k

DAY                = 1276          # снимок 30-дневного окна [1276, 1305]
EPOCHS             = 30
BATCH_SIZE         = 256
ENCODER_DIMS       = [600, 200]
DROPOUT            = 0.5
BETA               = 0.2
TOTAL_ANNEAL_STEPS = 200_000
KL_SCHEDULE        = "linear"
LR                 = 1e-3
WEIGHT_DECAY       = 0.0
BINARY             = True
MIN_USER_ITEMS     = 5
SEED               = 42


def build_history_from_events(
    train_lf: pl.LazyFrame,
    target_col: str,
    output_path: Path,
    binary: bool = True,
    min_interactions: int = 3,
    min_day: int | None = None,
) -> tuple[dict[str, int], dict[int, int]]:
    """Строит sparse user-history parquet из LazyFrame событий.

    Выходная схема: (user_idx: u32, user_id: u64, item_idx: list[u32], weight: list[f32]).
    train_lf должен уже быть отфильтрован по нужным item_id.

    Parameters
    ----------
    min_day : int or None
        Если задан, используются только события с ``day >= min_day``.
        Позволяет ограничить историю конкретным временным окном,
        аналогично снимку ``build_user_history_dataset(day=DAY)``.
    """
    if min_day is not None:
        train_lf = train_lf.filter(pl.col("day") >= min_day)

    df = (
        train_lf
        .select("user_id", "item_id", target_col)
        .collect(engine="streaming")
        .group_by("user_id", "item_id")
        .agg(pl.col(target_col).sum())
    )

    valid_users = (
        df.group_by("user_id").agg(pl.len().alias("cnt"))
        .filter(pl.col("cnt") >= min_interactions).select("user_id")
    )
    valid_items = (
        df.group_by("item_id").agg(pl.len().alias("cnt"))
        .filter(pl.col("cnt") >= min_interactions).select("item_id")
    )
    df = df.join(valid_users, on="user_id").join(valid_items, on="item_id")

    item_ids_sorted = df.select("item_id").unique().sort("item_id")["item_id"].to_list()
    item_to_idx: dict[str, int] = {iid: idx for idx, iid in enumerate(item_ids_sorted)}
    items_meta = pl.DataFrame(
        {"item_id": item_ids_sorted, "item_idx": list(range(len(item_ids_sorted)))},
        schema={"item_id": pl.String, "item_idx": pl.UInt32},
    )

    if binary:
        df = df.with_columns(weight=pl.lit(1.0, dtype=pl.Float32))
    else:
        df = df.with_columns(weight=pl.col(target_col).cast(pl.Float32))

    df = df.join(items_meta, on="item_id")

    user_df = (
        df.group_by("user_id")
        .agg(pl.col("item_idx"), pl.col("weight"))
        .sort("user_id")
        .with_row_index("user_idx")
        .with_columns(pl.col("user_idx").cast(pl.UInt32))
    )
    user_to_idx: dict[int, int] = dict(
        zip(user_df["user_id"].to_list(), user_df["user_idx"].to_list())
    )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    user_df.select("user_idx", "user_id", "item_idx", "weight").write_parquet(output_path)
    return item_to_idx, user_to_idx


def evaluate_ndcg(
    rec: MultiVAERecommender,
    test_lf: pl.LazyFrame,
    target_col: str,
    top_k: int = 15,
) -> dict:
    """Считает NDCG/Precision/Recall на тестовой выборке."""
    test_df = test_lf.select("user_id", "item_id", target_col).collect(engine="streaming")
    known_users = set(rec.user_to_idx.keys())
    test_filtered = test_df.filter(pl.col("user_id").is_in(known_users))

    user_test_truth = test_filtered.group_by("user_id").agg(
        pl.col("item_id").alias("true_items"),
        pl.col(target_col).alias("relevancy"),
    )
    test_user_ids = user_test_truth["user_id"].to_numpy()
    test_user_idxs = np.array([rec.user_to_idx[uid] for uid in test_user_ids])

    BATCH = 1024
    item_idx_chunks, score_chunks = [], []
    for start in range(0, len(test_user_idxs), BATCH):
        chunk = test_user_idxs[start : start + BATCH]
        ids, scores = rec.recommend_batch(chunk, top_k=top_k)
        item_idx_chunks.append(ids)
        score_chunks.append(scores)

    item_idx_matrix = np.concatenate(item_idx_chunks, axis=0)
    score_matrix    = np.concatenate(score_chunks,    axis=0)

    predicted_items_list  = [[rec.idx_to_item[i] for i in row] for row in item_idx_matrix]
    predicted_scores_list = [row.tolist() for row in score_matrix]

    prediction_df  = pl.DataFrame({
        "user_id":          test_user_ids,
        "predicted_items":  predicted_items_list,
        "predicted_scores": predicted_scores_list,
    })
    evaluation_df = user_test_truth.join(prediction_df, on="user_id")

    ndcg_results = ndcg_at_k(
        evaluation_df,
        relevancy_col="relevancy",
        true_items_col="true_items",
        predicted_items_col="predicted_items",
        predicted_score_col="predicted_scores",
        top_k=top_k,
    )
    mean_ndcg = float(ndcg_results["ndcg"].mean())
    precision, recall = precision_recall_at_k(
        evaluation_df,
        predicted_items_col="predicted_items",
        true_items_col="true_items",
        top_k=top_k,
    )
    return {"ndcg": mean_ndcg, "precision": precision, "recall": recall}


def optuna_objective(trial: optuna.Trial) -> float:
    target_col = trial.suggest_categorical("target_col", ["log_target", "sqrt_target"])
    latent_dim = trial.suggest_categorical("latent_dim", [64, 128, 200, 256])
    hidden_dim = trial.suggest_categorical("hidden_dim", [400, 600, 800])
    dropout    = trial.suggest_float("dropout", 0.2, 0.7, step=0.1)
    beta       = trial.suggest_float("beta", 0.05, 0.5, log=True)
    epochs     = trial.suggest_int("epochs", 10, 30, step=5)
    binary     = trial.suggest_categorical("binary", [True, False])

    tmp_dir = Path(tempfile.mkdtemp(prefix="vae_optuna_"))
    try:
        history_path  = tmp_dir / "history.pq"
        artifacts_dir = tmp_dir / "artifacts"

        # Шаг 1 — OOM-совместимый history parquet из train_sub
        # DAY — конец 30-дневного окна [DAY-29, DAY], как в снимке OOM baseline
        item_to_idx, user_to_idx = build_history_from_events(
            train_sub,
            target_col=target_col,
            output_path=history_path,
            binary=binary,
            min_interactions=3,
            min_day=DAY - 29,
        )

        # Шаг 2 — OOM-обучение (HuggingFace Arrow, dense батч только на шаге)
        train_vae(
            history_path=history_path,
            item_to_idx=item_to_idx,
            user_to_idx=user_to_idx,
            artifacts_dir=artifacts_dir,
            encoder_dims=[hidden_dim, latent_dim],
            dropout=dropout,
            beta=beta,
            epochs=epochs,
            batch_size=256,
            total_anneal_steps=100_000,
            use_lr_scheduler=True,
            num_workers=0,
            seed=RANDOM_SEED,
        )

        # Шаг 3 — загрузка и оценка на test_sub
        rec = MultiVAERecommender()
        rec.load(artifacts_dir / "model.pt")
        rec.set_user_items(load_npz(artifacts_dir / "user_items.npz"))

        metrics = evaluate_ndcg(rec, test_sub, target_col=target_col, top_k=TOP_K)
        ndcg = metrics["ndcg"]
        trial.set_user_attr("precision", metrics["precision"])
        trial.set_user_attr("recall",    metrics["recall"])
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    print(
        f"  Trial {trial.number}: target={target_col}, dims=[{hidden_dim},{latent_dim}], "
        f"dropout={dropout:.1f}, beta={beta:.3f}, epochs={epochs}, binary={binary} "
        f"→ NDCG@{TOP_K}={ndcg:.6f}"
    )
    return ndcg


sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)

print(f"Запуск Optuna: {OPTUNA_TRIALS} trials на {SUBSET_FRACTION*100:.0f}% датасета...")
study.optimize(optuna_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)

best = study.best_trial
print(f"\nЛучший trial #{best.number}:")
print(f"  NDCG@{TOP_K} = {best.value:.6f}")
for k, v in best.params.items():
    print(f"  {k} = {v}")

Запуск Optuna: 20 trials на 40% датасета...
2026-05-10 13:41:45.677 | INFO     | src.modeling.vae:build_model:403 - MultiVAE собрана: num_items=12148, dims=[600, 64], device=cuda


Generating train split: 0 examples [00:00, ? examples/s]

2026-05-10 13:41:45.854 | INFO     | src.modeling.train_vae:train_vae:424 - Стартую обучение: 169596 пользователей, 12148 items, batches per epoch ≈ 663


Эпохи:   3%|▎         | 1/30 [00:11<05:39, 11.69s/epoch, beta=0.000, kld=144.751, loss=77.969, lr=9.97e-04, nll=77.938]

2026-05-10 13:42:03.100 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 01: loss=77.9693 nll=77.9383 kld=144.7509 beta=0.0003 lr=9.97e-04 time=11.7s


...

2026-05-10 15:12:08.560 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 18: loss=70.9876 nll=69.9976 kld=125.8862 beta=0.0081 lr=3.42e-05 time=12.4s


Эпохи:  95%|█████████▌| 19/20 [03:55<00:12, 12.39s/epoch, beta=0.009, kld=124.704, loss=70.931, lr=1.61e-05, nll=69.894]

2026-05-10 15:12:20.940 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 19: loss=70.9306 nll=69.8938 kld=124.7036 beta=0.0085 lr=1.61e-05 time=12.4s


Эпохи: 100%|██████████| 20/20 [04:07<00:00, 12.39s/epoch, beta=0.009, kld=123.760, loss=71.003, lr=1.00e-05, nll=69.919]


2026-05-10 15:12:33.534 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 20: loss=71.0033 nll=69.9188 kld=123.7598 beta=0.0090 lr=1.00e-05 time=12.6s
2026-05-10 15:12:33.662 | INFO     | src.modeling.vae:save:497 - MultiVAE сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_optuna_qj1q1ytj\artifacts\model.pt
2026-05-10 15:12:34.438 | INFO     | src.modeling.train_vae:train_vae:549 - user_items.npz сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_optuna_qj1q1ytj\artifacts
2026-05-10 15:12:34.634 | INFO     | src.modeling.vae:build_model:403 - MultiVAE собрана: num_items=12148, dims=[600, 64], device=cuda
2026-05-10 15:12:34.643 | INFO     | src.modeling.vae:load:517 - MultiVAE загружена из C:\Users\Sasha\AppData\Local\Temp\vae_optuna_qj1q1ytj\artifacts\model.pt
  Trial 19: target=log_target, dims=[600,64], dropout=0.6, beta=0.068, epochs=20, binary=False → NDCG@15=0.208878

Лучший trial #14:
  NDCG@15 = 0.210775
  target_col = log_target
  latent_dim = 64
  hidden_dim = 600
  d

In [8]:
best = study.best_trial
print(f"\nЛучший trial #{best.number}:")
print(f"  NDCG@{TOP_K} = {best.value:.6f}")
for k, v in best.params.items():
    print(f"  {k} = {v}")


Лучший trial #14:
  NDCG@15 = 0.210775
  target_col = log_target
  latent_dim = 64
  hidden_dim = 600
  dropout = 0.7
  beta = 0.10359660012799471
  epochs = 10
  binary = False


In [12]:
best_params = {
    "target_col": "log_target",
    "latent_dim": 64,
    "hidden_dim": 600,
    "dropout": 0.7,
    "beta": 0.10359660012799471,
    "epochs": 10,
    "binary": False,
}

In [9]:
trials_df = pl.DataFrame([
    {
        "trial":      t.number,
        "ndcg":       t.value,
        "precision":  t.user_attrs.get("precision"),
        "recall":     t.user_attrs.get("recall"),
        "target_col": t.params["target_col"],
        "latent_dim": t.params["latent_dim"],
        "hidden_dim": t.params["hidden_dim"],
        "dropout":    t.params["dropout"],
        "beta":       t.params["beta"],
        "epochs":     t.params["epochs"],
        "binary":     t.params["binary"],
    }
    for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
]).sort("ndcg", descending=True)

trials_df

trial,ndcg,precision,recall,target_col,latent_dim,hidden_dim,dropout,beta,epochs,binary
i64,f64,f64,f64,str,i64,i64,f64,f64,i64,bool
14,0.210775,0.057536,0.041814,"""log_target""",64,600,0.7,0.103597,10,false
11,0.210711,0.057509,0.041778,"""log_target""",64,600,0.7,0.113721,10,false
10,0.21069,0.057508,0.041774,"""log_target""",64,600,0.7,0.117938,10,false
17,0.210578,0.055666,0.040249,"""log_target""",64,600,0.5,0.099063,15,false
15,0.209448,0.055892,0.040399,"""log_target""",64,600,0.6,0.081171,15,false
…,…,…,…,…,…,…,…,…,…,…
6,0.204295,0.05303,0.037863,"""sqrt_target""",200,800,0.5,0.295078,20,true
4,0.203279,0.052967,0.038299,"""sqrt_target""",200,400,0.5,0.069166,30,false
0,0.202504,0.053189,0.038063,"""sqrt_target""",64,600,0.6,0.052427,30,true


## Финальная оценка лучшего Mult-VAE на полном датасете

Переобучаем модель с найденными гиперпараметрами на **полном** train/test split — для прямого сравнения с iALS и SVD.

In [13]:
# best_params = best.params
print("Лучшие гиперпараметры (из hyperopt на subset):")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# Фильтруем train по тем же 40k item-ам и тому же 30-дневному окну [DAY-29, DAY],
# что и в Optuna — чтобы финальная модель была сравнима с гиперопт-триалами
train_full = train.filter(pl.col("item_id").is_in(all_items_ids))

print("\nОбучение на полном датасете (OOM)...")
_tmp_dir = Path(tempfile.mkdtemp(prefix="vae_final_"))
try:
    _history_path  = _tmp_dir / "history.pq"
    _artifacts_dir = _tmp_dir / "artifacts"

    _item_to_idx, _user_to_idx = build_history_from_events(
        train_full,
        target_col=best_params["target_col"],
        output_path=_history_path,
        binary=best_params["binary"],
        min_interactions=5,
        min_day=DAY - 29,
    )
    print(f"  {len(_user_to_idx):,} пользователей x {len(_item_to_idx):,} items")

    train_vae(
        history_path=_history_path,
        item_to_idx=_item_to_idx,
        user_to_idx=_user_to_idx,
        artifacts_dir=_artifacts_dir,
        encoder_dims=[best_params["hidden_dim"], best_params["latent_dim"]],
        dropout=best_params["dropout"],
        beta=best_params["beta"],
        epochs=best_params["epochs"],
        batch_size=512,
        total_anneal_steps=200_000,
        use_lr_scheduler=True,
        num_workers=0,
        seed=SEED,
    )

    _rec = MultiVAERecommender()
    _rec.load(_artifacts_dir / "model.pt")
    _rec.set_user_items(load_npz(_artifacts_dir / "user_items.npz"))

    _metrics = evaluate_ndcg(_rec, test, target_col=best_params["target_col"], top_k=TOP_K)
finally:
    shutil.rmtree(_tmp_dir, ignore_errors=True)

best_vae_result = {
    "target":       best_params["target_col"],
    "encoder_dims": [best_params["hidden_dim"], best_params["latent_dim"]],
    "dropout":      best_params["dropout"],
    "beta":         best_params["beta"],
    "epochs":       best_params["epochs"],
    "top_k":        TOP_K,
    "ndcg":         _metrics["ndcg"],
    "precision":    _metrics["precision"],
    "recall":       _metrics["recall"],
    "rmse":         None,
    "mae":          None,
}

print("\nРезультат лучшего Mult-VAE (full data):")
for metric in ("ndcg", "precision", "recall"):
    print(f"  {metric}: {best_vae_result[metric]:.6f}")

Лучшие гиперпараметры (из hyperopt на subset):
  target_col: log_target
  latent_dim: 64
  hidden_dim: 600
  dropout: 0.7
  beta: 0.10359660012799471
  epochs: 10
  binary: False

Обучение на полном датасете (OOM)...
  350,650 пользователей x 12,866 items
2026-05-10 18:51:44.940 | INFO     | src.modeling.vae:build_model:403 - MultiVAE собрана: num_items=12866, dims=[600, 64], device=cuda


Generating train split: 0 examples [00:00, ? examples/s]

2026-05-10 18:51:45.072 | INFO     | src.modeling.train_vae:train_vae:424 - Стартую обучение: 350650 пользователей, 12866 items, batches per epoch ≈ 685


Эпохи:  10%|█         | 1/10 [00:23<03:35, 23.91s/epoch, beta=0.000, kld=163.004, loss=106.890, lr=9.76e-04, nll=106.855]

2026-05-10 18:52:18.392 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 01: loss=106.8904 nll=106.8547 kld=163.0041 beta=0.0004 lr=9.76e-04 time=23.9s


...

2026-05-10 18:56:28.822 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 08: loss=87.8742 nll=87.4060 kld=175.9827 beta=0.0028 lr=1.05e-04 time=41.4s


Эпохи:  90%|█████████ | 9/10 [05:16<00:39, 39.71s/epoch, beta=0.003, kld=172.485, loss=87.676, lr=3.42e-05, nll=87.156]

2026-05-10 18:57:11.087 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 09: loss=87.6757 nll=87.1555 kld=172.4847 beta=0.0032 lr=3.42e-05 time=42.3s


Эпохи: 100%|██████████| 10/10 [05:58<00:00, 35.86s/epoch, beta=0.004, kld=170.460, loss=87.613, lr=1.00e-05, nll=87.039]


2026-05-10 18:57:53.121 | INFO     | src.modeling.train_vae:train_vae:527 - Epoch 10: loss=87.6133 nll=87.0387 kld=170.4597 beta=0.0035 lr=1.00e-05 time=42.0s
2026-05-10 18:57:53.437 | INFO     | src.modeling.vae:save:497 - MultiVAE сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_final_l006mvtd\artifacts\model.pt
2026-05-10 18:57:56.250 | INFO     | src.modeling.train_vae:train_vae:549 - user_items.npz сохранена в C:\Users\Sasha\AppData\Local\Temp\vae_final_l006mvtd\artifacts
2026-05-10 18:57:56.814 | INFO     | src.modeling.vae:build_model:403 - MultiVAE собрана: num_items=12866, dims=[600, 64], device=cuda
2026-05-10 18:57:56.827 | INFO     | src.modeling.vae:load:517 - MultiVAE загружена из C:\Users\Sasha\AppData\Local\Temp\vae_final_l006mvtd\artifacts\model.pt

Результат лучшего Mult-VAE (full data):
  ndcg: 0.217632
  precision: 0.060075
  recall: 0.040805


## Сравнение Mult-VAE / iALS / SVD

Условия одинаковые для всех:

- **Данные**: marketplace events, global temporal split 80/20
- **Метрика ранжирования**: NDCG@15, Precision@15, Recall@15
- **SVD**: лучший результат из `svd_experiments_unified.ipynb`
- **iALS**: лучший результат из `ials_research.ipynb`
- **Mult-VAE**: конфигурация, найденная Optuna на 40%-подмножестве

In [14]:
svd_best = {
    "model":     "SVD (best)",
    "target":    "log_target",
    "top_k":     15,
    "ndcg":      0.135736,
    "precision": 0.026741,
    "recall":    0.026168,
    "rmse":      3.111453,
    "mae":       0.994588,
    "hyperopt":  "grid (ручной)",
}

ials_best = {
    "model":     "iALS (best, Optuna)",
    "target":    "log_target",
    "top_k":     15,
    "ndcg":      0.144,         # подставить актуальное из ials_research
    "precision": 0.027,         # подставить актуальное из ials_research
    "recall":    0.030,         # подставить актуальное из ials_research
    "rmse":      None,
    "mae":       None,
    "hyperopt":  "Optuna TPE, 25 trials, 40% subset",
}

vae_best = {
    "model":     "Mult-VAE (best, Optuna)",
    "target":    best_vae_result["target"],
    "top_k":     best_vae_result["top_k"],
    "ndcg":      best_vae_result["ndcg"],
    "precision": best_vae_result["precision"],
    "recall":    best_vae_result["recall"],
    "rmse":      best_vae_result["rmse"],
    "mae":       best_vae_result["mae"],
    "hyperopt":  f"Optuna TPE, {OPTUNA_TRIALS} trials, {SUBSET_FRACTION*100:.0f}% subset",
}

final_comparison = pl.DataFrame([svd_best, ials_best, vae_best])

ndcg_delta_vs_svd  = vae_best["ndcg"] - svd_best["ndcg"]
ndcg_delta_vs_ials = vae_best["ndcg"] - ials_best["ndcg"]
print(f"NDCG@15: SVD={svd_best['ndcg']:.4f}  iALS={ials_best['ndcg']:.4f}  Mult-VAE={vae_best['ndcg']:.4f}")
print(f"  Δ vs SVD:  {ndcg_delta_vs_svd:+.4f} ({ndcg_delta_vs_svd/svd_best['ndcg']*100:+.1f}%)")
print(f"  Δ vs iALS: {ndcg_delta_vs_ials:+.4f} ({ndcg_delta_vs_ials/ials_best['ndcg']*100:+.1f}%)")

final_comparison

NDCG@15: SVD=0.1357  iALS=0.1440  Mult-VAE=0.2176
  Δ vs SVD:  +0.0819 (+60.3%)
  Δ vs iALS: +0.0736 (+51.1%)


model,target,top_k,ndcg,precision,recall,rmse,mae,hyperopt
str,str,i64,f64,f64,f64,f64,f64,str
"""SVD (best)""","""log_target""",15,0.135736,0.026741,0.026168,3.111453,0.994588,"""grid (ручной)"""
"""iALS (best, Optuna)""","""log_target""",15,0.144,0.027,0.03,null,null,"""Optuna TPE, 25 trials, 40% sub…"
"""Mult-VAE (best, Optuna)""","""log_target""",15,0.217632,0.060075,0.040805,null,null,"""Optuna TPE, 20 trials, 40% sub…"


## Выводы

- Mult-VAE учит нелинейное представление user-history и работает не на скалярном произведении, а на мультикатегориальном-распределении по items — это даёт ему преимущество на длинном хвосте.
- KL-annealing ($\beta$ от 0 до целевого) важен — без него VAE быстро коллапсирует в обычный AE.
- Логарифмированный таргет все еще лучше всего показывает себя в контексте качества модели. Также на текущий момент на MultVAE показывает наилучшие результаты по целевой метрике - ndcg@15.